In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from eng_quant.ipm import ipm_solve_no_split
from eng_quant.generators import toy_problem_to_json
from eng_quant.relaxers import (
    quadratic_lin_approx_no_surrogate,
    quadratic_lin_approx_no_surrogate_refine
)
from eng_quant.testers import (
    solve_lp_return_x,
    solve_qcp_return_x,
    condition_number_nes_basic
)

import matplotlib.pyplot as plt
import numpy as np

In [21]:
# Number of copies of the network in the toy system
N=5

# Whether the production and flow variables have upper bounds
F_UPPER_BOUNDS=True

# Whether the demand constraint is an equality or inequality
DEMAND_INEQUALITY=True

# Capacity / demand parameters
# GAMMA=1.0
GAMMA=1.0
LAMBD=1.0
EPS=0.1
DELTA=0.1

# Lower bounds and upper bound on pressure
LX=LP=LF=0.0
UP=0.5

LIN_APPROX = 5

# Whether to do an inner or outer approximation, THIS SHOULD STAY TRUE HERE
OUTER_APPROXIMATION = True

# Whether the coefficient is part of the surrogate
COEFFICIENT_SURROGATE = True

# Whether to bound the surrogates with bound constraints.
SURROGATE_BOUND_BELOW = True
SURROGATE_BOUND_ABOVE = True

# Whether to divide by lambda in an inner approximation
REMOVE_DIVISION = False

In [22]:
toy_problem_to_json(N, F_UPPER_BOUNDS, DEMAND_INEQUALITY, GAMMA, LAMBD,
                        EPS, DELTA, LP, LF, UP)
quad, q_time, q_x = solve_qcp_return_x(f_name="toy.json")
print(f"Exact objective value: {quad}")
print(f"Exact solution: {q_x}")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26300.2))

CPU model: AMD Ryzen 7 8845HS w/ Radeon 780M Graphics, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Presolve a model with 0 rows, 49 columns and 0 nonzeros (Min)
Model fingerprint: 0xd9911d43
Model has 15 linear objective coefficients
Model has 34 quadratic constraints
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  QMatrix range    [1e+00, 1e+00]
  QLMatrix range   [1e+00, 1e+00]
  Objective range  [9e-01, 1e+00]
  Bounds range     [5e-01, 1e+00]
  RHS range        [0e+00, 0e+00]
  QRHS range       [1e+00, 1e+00]
Presolve removed 0 rows and 15 columns
Statistics for model 'qcp_from_json_pre':
  Problem type                : QCP
  Linear constraint matrix    : 82 rows, 82 columns, 162 nonzeros
  Variable types              : 82 continuous, 0 integer (0 binary)
  Quadratic constraints       : 24 constrs, 72 nonzeros
  Ma

In [23]:
quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION, remove_division=REMOVE_DIVISION,
                                    f_name="toy.json", endpoints=False)
val, time, x, compl = ipm_solve_no_split(f_name="linear_approx.json")
print(x)
print(condition_number_nes_basic())

print(f"Error: {val - quad}")

Solution is below target precision.
The algorithm stopped after 40 iterations in 0.04 seconds.

Primal objective:        -5.54872384e+00
Dual objective:          -5.54872385e+00

Norm of Primal residual: 1.68e-13
Norm of Dual residual:   6.69e-14
Abs. Compl. Gap:         4.35e-09
Rel. Compl. Gap:         3.60e-10

[5.00000000e-01 4.57709414e-01 4.66407565e-01 5.00000000e-01 4.57122128e-01 4.66473957e-01 5.00000000e-01 4.57178692e-01 4.66502268e-01 5.00000000e-01 4.57168105e-01 4.66533579e-01 5.00000000e-01
 4.56461818e-01 4.66674609e-01 2.77777776e-01 2.77777776e-01 2.77777776e-01 2.77777776e-01 2.77777776e-01 1.93654632e-10 1.95156800e-10 1.95467736e-10 1.95704189e-10 1.96444191e-10 4.72222224e-01
 7.46228596e-02 5.69669917e-01 5.30330083e-01 4.72222224e-01 5.95761644e-02 5.69669917e-01 5.30330083e-01 1.65149999e-02 4.72222224e-01 5.82326296e-02 5.69669917e-01 5.30330083e-01 1.79833046e-02
 4.72222224e-01 5.64482460e-02 5.69669917e-01 5.30330083e-01 1.81080745e-02 4.72222223e-01 4.165

In [24]:
# Define a set of functions for each

quads = [[val] for val in x[(5 * N):]]
# print(quads)
iters = 5

for i in range(1, iters + 1):
    print(f"Iteration {i}")
    points_functions = []

    def np_uniform_add_q_factory(qs):
        def np_uniform_add_qs(lower, upper, num):
            refinement_points = len(qs)
            uniform = np.linspace(lower, upper, num - refinement_points)
            for q in qs:
                idx = np.searchsorted(uniform, q)
                uniform = np.insert(uniform, idx, q)
            return uniform
        return np_uniform_add_qs

    for qs in quads:
        points_functions.append(np_uniform_add_q_factory(qs))

    quadratic_lin_approx_no_surrogate_refine(LIN_APPROX + i, OUTER_APPROXIMATION, points_function=points_functions,
                                                remove_division=REMOVE_DIVISION, f_name="toy.json")
    # quadratic_lin_approx_no_surrogate(LIN_APPROX, OUTER_APPROXIMATION,
    #                                             remove_division=REMOVE_DIVISION, f_name="toy.json", endpoints=False)

    val, time, x, compl = ipm_solve_no_split(f_name="linear_approx.json")

    # print(f"x: {x}")
    print(f"val: {val}")
    print(f"Error: {val - quad}")

    for j, qs in enumerate(quads):
        qs.append(x[(5 * N) + j])

    print(quads)

# print(x)
# print(f"Error: {val - quad}")

Iteration 1
Solution is below target precision.
The algorithm stopped after 43 iterations in 0.05 seconds.

Primal objective:        -5.54955358e+00
Dual objective:          -5.54955358e+00

Norm of Primal residual: 2.19e-13
Norm of Dual residual:   7.65e-14
Abs. Compl. Gap:         4.55e-09
Rel. Compl. Gap:         3.76e-10

val: 5.549553583643775
Error: -0.00044887638333701574
[[np.float64(0.47222222374619155), np.float64(0.5004464032714349)], [np.float64(0.07462285958345201), np.float64(4.998178853257713e-08)], [np.float64(0.5696699166166742), np.float64(0.5995535706709902)], [np.float64(0.5303300834011448), np.float64(0.5004464293458557)], [np.float64(0.47222222371322076), np.float64(0.5004464065479983)], [np.float64(0.05957616438966305), np.float64(1.5949674608268315e-08)], [np.float64(0.5696699165884626), np.float64(0.5995535737633687)], [np.float64(0.5303300834293588), np.float64(0.5004464262534832)], [np.float64(0.016514999928498728), np.float64(2.390736777532276e-08)], [np.flo